In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt

# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1)
images = mnist.data.to_numpy().reshape(-1, 28, 28)
labels = mnist.target.astype(int)

eights = images[labels == 8] #select digit 8

# Show a few digit 8s to verify
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(eights[i], cmap='gray')
    plt.axis('off')

plt.show()

def extract_patches(images, patch_size=9):
  half = patch_size // 2 #the patch will extend {half} pixels in each direction from the center
  patch_list = []

  for img in images:
    for i in range(half, 28 - half):
      for j in range(half, 28 - half):
        patch = img[i - half:i + half + 1, j - half:j + half + 1] #9 * 9 patch centered at pixel (i, j)
        patch_list.append(patch)
  return np.stack(patch_list)

patch_library = extract_patches(eights, patch_size=9)
for k in range(5):
    plt.subplot(1, 5, k + 1)
    plt.imshow(patch_library[k], cmap='gray')
    plt.axis('off')

plt.show()
patch_library.shape

output_size = 28
patch_size = 9
half = patch_size // 2
output = np.zeros((output_size, output_size), dtype = np.uint8) #initialize output image
filled_mask = np.zeros((output_size, output_size), dtype = bool) #initialize filled mask
seed_patch = patch_library[np.random.randint(len(patch_library))] #randomly select a seed patch

center =output_size // 2
output[center - half:center + half + 1, center - half:center + half + 1] = seed_patch #place the seed patch to the center of the output image
filled_mask[center - half:center + half + 1, center - half:center + half + 1] = True
print("Initial filled pixels:", np.sum(filled_mask))

plt.imshow(output, cmap='gray')
plt.title("Seed Patch Initialized (9x9)")
plt.axis('off')
plt.show()

def find_frontier(filled_mask, window_size):
  frontier = []
  half = window_size // 2
  height, width = filled_mask.shape

  for y in range(height):
    for x in range(width):
      if filled_mask[y, x]: #if (y, x) is already filled, continue, no need to check if there's any other context
        continue

      #define window boundary
      y_start = max(0, y - half)
      y_end = min(height, y + half + 1)
      x_start = max(0, x - half)
      x_end = min(width, x + half + 1)

      window = filled_mask[y_start:y_end, x_start:x_end]
      if np.any(window): #check if there's any thing in this (y, x) centered window
        frontier.append((y, x))

  return frontier

def rank_frontier(filled_mask, frontier_pixels, window_size):
    half = window_size // 2
    h, w = filled_mask.shape
    ranked = []

    for y, x in frontier_pixels:
        y_start = max(0, y - half)
        y_end = min(h, y + half + 1)
        x_start = max(0, x - half)
        x_end = min(w, x + half + 1)

        context_window = filled_mask[y_start:y_end, x_start:x_end]
        context_score = np.sum(context_window)  # count filled pixels
        ranked.append(((y, x), context_score))

    # Sort by score descending (more context → higher priority)
    ranked.sort(key=lambda t: -t[1])
    return [pos for pos, _ in ranked]

def gaussian_weight(window_size):
  half = window_size // 2
  y, x = np.mgrid[-half:half + 1, -half:half + 1] #create two 9 * 9 grids range from -4 to 4 to determine the distance between each pixel and the center(y, x)
  sigma = window_size / 6.4
  g = np.exp(-(x**2 + y**2) / (2 * sigma**2))
  return g / g.sum()

def matching(y, x, output, filled_mask, patch_library, window_size, gaussian_weight):
  half = window_size // 2
  h, w = output.shape
  best_patches = []
  minimum_error = float('inf')

  y_start = max(0, y - half)
  y_end = min(h, y + half + 1)
  x_start = max(0, x - half)
  x_end = min(w, x + half + 1)

  output_window = output[y_start:y_end, x_start:x_end] #frontier(y, x) centered window to look the context around it
  mask_window = filled_mask[y_start:y_end, x_start:x_end] #tells which values are already filled

  if not np.any(mask_window): #ensures there's context
    return None

  #align gaussian mask
  gw_y_start = half - (y - y_start)
  gw_y_end = gw_y_start + (y_end - y_start)
  gw_x_start = half - (x - x_start)
  gw_x_end = gw_x_start + (x_end - x_start)
  gaussian_crop = gaussian_weight[gw_y_start:gw_y_end, gw_x_start:gw_x_end]

  for patch in patch_library:
    patch_window = patch[gw_y_start:gw_y_end, gw_x_start:gw_x_end]
    difference = (patch_window - output_window) * mask_window
    ssd = np.sum(gaussian_crop * (difference**2))

    if ssd < minimum_error:
      minimum_error = ssd
      best_patches = [patch]
    elif ssd <= minimum_error * 1.1:
      best_patches.append(patch)

  if not best_patches:
    return None

  chosen_patch = best_patches[np.random.randint(len(best_patches))]
  return chosen_patch[half, half]

g_weight = gaussian_weight(patch_size)
total_pixels = output_size * output_size
filled_count = np.sum(filled_mask)
iteration = 0

while filled_count < total_pixels:
  iteration += 1
  print(f"Iteration {iteration}: Filled {filled_count}/{total_pixels}")

  frontier_pixels = find_frontier(filled_mask, patch_size)
  frontier_pixels = rank_frontier(filled_mask, frontier_pixels, patch_size)
  filled_this_round = False

  for y, x in frontier_pixels:
      pixel_value = matching(y, x, output, filled_mask, patch_library, patch_size, g_weight)

      if pixel_value is not None:
          output[y, x] = pixel_value
          filled_mask[y, x] = True
          filled_count += 1
          filled_this_round = True
          break

  if not filled_this_round:
        print("No match found — try changing patch size or seed.")
        break


  if iteration % 10 == 0 or filled_count == 784:
      plt.imshow(output, cmap='gray')
      plt.title(f"Final Synthesized Digit")
      plt.axis('off')
      plt.show()